## Treshold Optimization

In [1]:
#!/usr/bin/env python3
"""
================================================================================
RAMP-ML -- GNN Threshold Search (Inferenza e Ottimizzazione Soglia am[i])
================================================================================
Questo script è autosufficiente: non fa training, carica il modello salvato 
e spazza le soglie di probabilità (da 0.10 a 0.95) per trovare quella che 
massimizza l'Accuracy e l'F1-Score sulla classificazione della tecnologia.
================================================================================
"""

import json, glob, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# PyTorch Geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv, Linear

# ======================================================================
# CONFIGURAZIONE (Controlla i percorsi)
# ======================================================================
DATASET_DIR = r"C:\Users\stefa\OneDrive\Desktop\MAGISTRALE\TESI\TEST CODICE\GNN dataset\gnn_dataset_v2_TEST"
MODEL_PATH  = "gnn_model.pt"  # <-- Assicurati che il file sia nella stessa cartella o metti il path assoluto
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SEED        = 42

# Parametri architetturali di default (verranno sovrascritti se salvati nel dict del modello)
HIDDEN_DIM  = 128
NUM_LAYERS  = 2
DROPOUT     = 0.2

# ======================================================================
# 1. FUNZIONI DI CARICAMENTO DATI
# ======================================================================
def load_instance(fpath):
    with open(fpath) as f:
        d = json.load(f)

    nI, nD, nC = d['nI'], d['nD'], d['nC']

    ratio   = np.array(d['ratio'], dtype=np.float32)
    c_CM    = np.array(d['c_CM'], dtype=np.float32)
    L_CM    = np.array(d['L_CM'], dtype=np.float32)
    L_AM    = np.array(d['L_AM'], dtype=np.float32)
    b_i     = np.array(d['b_i'], dtype=np.float32)
    SL      = np.array(d['SL'], dtype=np.float32)
    netto   = np.array(d['netto'], dtype=np.float32)
    mu_ic   = np.array(d['mu_ic'], dtype=np.float32)
    mu_tot  = mu_ic.sum(axis=1)

    log_c_CM = np.log10(np.clip(c_CM, 1, None))
    log_b_i  = np.log10(np.clip(b_i, 1, None))
    netto_norm = netto / (np.abs(netto).max() + 1e-8)

    coord_DC_raw = np.array(d['coord_DC'], dtype=np.float32)
    coord_C_raw  = np.array(d['coord_C'], dtype=np.float32)

    mu_sum = mu_tot[:, None] + 1e-8
    baricentro = (mu_ic @ coord_C_raw) / mu_sum

    diff = coord_C_raw[None, :, :] - baricentro[:, None, :]
    dist2 = (diff ** 2).sum(axis=2)
    dispersione = (mu_ic * dist2).sum(axis=1) / mu_sum[:, 0]

    diff_dc = coord_DC_raw[None, :, :] - baricentro[:, None, :]
    dist_dc = np.sqrt((diff_dc ** 2).sum(axis=2))
    dist_dc_min  = dist_dc.min(axis=1)
    dist_dc_max  = dist_dc.max(axis=1)
    dist_dc_mean = dist_dc.mean(axis=1)
    dist_dc_std  = dist_dc.std(axis=1)

    DIAG_MAX = np.sqrt(200.0)
    baricentro_norm    = baricentro / 10.0
    dispersione_norm   = np.sqrt(dispersione) / DIAG_MAX
    dist_dc_min_norm   = dist_dc_min / DIAG_MAX
    dist_dc_max_norm   = dist_dc_max / DIAG_MAX
    dist_dc_mean_norm  = dist_dc_mean / DIAG_MAX
    dist_dc_std_norm   = dist_dc_std / DIAG_MAX

    sku_features = np.stack([
        ratio / 6.0, log_c_CM / 4.0, L_CM * 52 / 35.0, L_AM * 52 / 4.0,
        log_b_i / 5.5, SL, netto_norm, mu_tot / (mu_tot.max() + 1e-8),
        np.full(nI, d['Leas'] / 30000.0, dtype=np.float32),
        np.full(nI, d['costo_km'] / 200.0, dtype=np.float32),
        baricentro_norm[:, 0], baricentro_norm[:, 1], dispersione_norm,
        dist_dc_min_norm, dist_dc_max_norm, dist_dc_mean_norm, dist_dc_std_norm
    ], axis=1)

    dc_features = np.array(d['coord_DC'], dtype=np.float32) / 10.0
    cli_features = np.array(d['coord_C'], dtype=np.float32) / 10.0

    src_demands, dst_demands = [], []
    for i in range(nI):
        for c in range(nC):
            if mu_ic[i, c] > 1e-6:
                src_demands.append(c)
                dst_demands.append(i)

    dist_dc_c = np.array(d['dist_dc_c'], dtype=np.float32)
    src_prox, dst_prox = [], []
    for dd in range(nD):
        for c in range(nC):
            src_prox.append(dd)
            dst_prox.append(c)

    src_port, dst_port = [], []
    for i in range(nI):
        for j in range(i+1, nI):
            src_port.extend([i, j])
            dst_port.extend([j, i])

    am_label = torch.tensor(d['am'], dtype=torch.float32)
    dc_label = torch.tensor(d['dc'], dtype=torch.float32)
    y_label_raw = np.array(d['y'], dtype=np.int64)

    y_class = torch.zeros(nI, nC, dtype=torch.long)
    for i in range(nI):
        for c in range(nC):
            assigned = np.where(y_label_raw[:, i, c] == 1)[0]
            y_class[i, c] = int(assigned[0]) if len(assigned) > 0 else 0

    data = HeteroData()
    data['sku'].x = torch.tensor(sku_features, dtype=torch.float32)
    data['dc'].x  = torch.tensor(dc_features, dtype=torch.float32)
    data['cli'].x = torch.tensor(cli_features, dtype=torch.float32)

    if src_demands:
        data['cli', 'demands', 'sku'].edge_index = torch.tensor([src_demands, dst_demands], dtype=torch.long)
        data['sku', 'demanded_by', 'cli'].edge_index = torch.tensor([dst_demands, src_demands], dtype=torch.long)
    data['dc', 'near', 'cli'].edge_index = torch.tensor([src_prox, dst_prox], dtype=torch.long)
    data['cli', 'near_rev', 'dc'].edge_index = torch.tensor([dst_prox, src_prox], dtype=torch.long)
    if src_port:
        data['sku', 'portfolio', 'sku'].edge_index = torch.tensor([src_port, dst_port], dtype=torch.long)

    data['sku'].am_label = am_label
    data.dc_label = dc_label
    data.y_class  = y_class
    data.nI = nI
    data.nD = nD
    data.nC = nC

    return data

def load_dataset(dataset_dir):
    files = sorted(glob.glob(os.path.join(dataset_dir, "instance_*.json")))
    dataset = []
    for f in files:
        try:
            dataset.append(load_instance(f))
        except Exception as e:
            print(f"Errore su {os.path.basename(f)}: {e}")
    return dataset

# ======================================================================
# 2. DEFINIZIONE MODELLO GNN (Stessa struttura per caricare i pesi)
# ======================================================================
class HeteroGNN(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.proj_sku = Linear(17, hidden_dim)
        self.proj_dc  = Linear(2, hidden_dim)
        self.proj_cli = Linear(2, hidden_dim)
        self.dropout = dropout

        self.convs = nn.ModuleList()
        self.norms_sku = nn.ModuleList()
        self.norms_dc  = nn.ModuleList()
        self.norms_cli = nn.ModuleList()

        for _ in range(num_layers):
            conv = HeteroConv({
                ('cli', 'demands', 'sku'):     SAGEConv(hidden_dim, hidden_dim),
                ('sku', 'demanded_by', 'cli'): SAGEConv(hidden_dim, hidden_dim),
                ('dc', 'near', 'cli'):         SAGEConv(hidden_dim, hidden_dim),
                ('cli', 'near_rev', 'dc'):     SAGEConv(hidden_dim, hidden_dim),
                ('sku', 'portfolio', 'sku'):    SAGEConv(hidden_dim, hidden_dim),
            }, aggr='sum')
            self.convs.append(conv)
            self.norms_sku.append(nn.LayerNorm(hidden_dim))
            self.norms_dc.append(nn.LayerNorm(hidden_dim))
            self.norms_cli.append(nn.LayerNorm(hidden_dim))

        self.head_am = nn.Sequential(
            Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            Linear(64, 1)
        )
        self.head_dc = nn.Sequential(
            Linear(hidden_dim * 2, 64), nn.ReLU(), nn.Dropout(dropout),
            Linear(64, 1)
        )
        self.head_y = nn.Sequential(
            Linear(hidden_dim * 3 + 1, 64), nn.ReLU(), nn.Dropout(dropout),
            Linear(64, 1)
        )

    def forward(self, data):
        x_dict = {
            'sku': self.proj_sku(data['sku'].x),
            'dc':  self.proj_dc(data['dc'].x),
            'cli': self.proj_cli(data['cli'].x),
        }
        for i, conv in enumerate(self.convs):
            x_dict_new = conv(x_dict, data.edge_index_dict)
            for ntype in x_dict:
                if ntype in x_dict_new:
                    h = x_dict_new[ntype]
                    if ntype == 'sku': h = self.norms_sku[i](h)
                    elif ntype == 'dc': h = self.norms_dc[i](h)
                    elif ntype == 'cli': h = self.norms_cli[i](h)
                    h = F.relu(h)
                    h = F.dropout(h, p=self.dropout, training=self.training)
                    x_dict[ntype] = x_dict[ntype] + h 
        return x_dict

    def predict_am(self, x_dict):
        return self.head_am(x_dict['sku']).squeeze(-1)

# ======================================================================
# 3. ESECUZIONE (Caricamento e Analisi)
# ======================================================================
if __name__ == '__main__':
    print(f"Device in uso: {DEVICE}")
    print("Caricamento dataset in corso...")
    dataset = load_dataset(DATASET_DIR)
    
    # 3.1 Riproduciamo lo split originale per avere lo stesso test_data
    indices = list(range(len(dataset)))
    strat_labels = [d.nI for d in dataset]
    train_idx, temp_idx = train_test_split(indices, test_size=0.30, stratify=strat_labels, random_state=SEED)
    temp_strat = [strat_labels[i] for i in temp_idx]
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=temp_strat, random_state=SEED)
    
    test_data = [dataset[i] for i in test_idx]
    print(f"Test data ricreato: {len(test_data)} istanze pronte per l'analisi.")

    # 3.2 Caricamento del Modello
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"File modello non trovato: {MODEL_PATH}")
        
    print(f"Caricamento modello da {MODEL_PATH}...")
    ckpt = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
    
    # Recuperiamo la configurazione originale usata in fase di train se esiste
    cfg = ckpt.get('config', {})
    h_dim = cfg.get('hidden_dim', HIDDEN_DIM)
    n_lay = cfg.get('num_layers', NUM_LAYERS)
    drop  = cfg.get('dropout', DROPOUT)

    model = HeteroGNN(hidden_dim=h_dim, num_layers=n_lay, dropout=drop).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print("Modello caricato con successo.")

    # 3.3 Ricerca della soglia ottimale
    print("\n" + "="*60)
    print("RICERCA SOGLIA OTTIMALE PER 'am' SUL TEST SET")
    print("="*60)

    all_am_probs = []
    all_am_true = []

    # Singola passata per raccogliere probabilità e etichette reali
    with torch.no_grad():
        for data in test_data:
            data = data.to(DEVICE)
            x_dict = model(data)
            am_logits = model.predict_am(x_dict)
            
            # Funzione Sigmoide per ottenere probabilità [0, 1]
            am_probs = torch.sigmoid(am_logits)
            
            all_am_probs.extend(am_probs.cpu().tolist())
            all_am_true.extend(data['sku'].am_label.cpu().tolist())

    all_am_probs = np.array(all_am_probs)
    all_am_true = np.array(all_am_true)

    # Scansione soglie
    thresholds = np.arange(0.1, 0.95, 0.05)
    best_thresh_acc, best_acc = 0.5, 0.0
    best_thresh_f1, best_f1 = 0.5, 0.0

    print(f"{'Soglia':>10} | {'Accuracy':>10} | {'F1-Score':>10}")
    print("-" * 37)

    for t in thresholds:
        am_pred = (all_am_probs >= t).astype(float)
        acc = accuracy_score(all_am_true, am_pred)
        f1 = f1_score(all_am_true, am_pred, zero_division=0)
        
        print(f"{t:10.2f} | {acc:10.4f} | {f1:10.4f}")
        
        if acc > best_acc:
            best_acc = acc
            best_thresh_acc = t
        if f1 > best_f1:
            best_f1 = f1
            best_thresh_f1 = t

    print("-" * 37)
    print(f"Soglia base 0.50 -> Acc: {accuracy_score(all_am_true, (all_am_probs >= 0.5).astype(float)):.4f} | F1: {f1_score(all_am_true, (all_am_probs >= 0.5).astype(float), zero_division=0):.4f}")
    print(f"Miglior soglia per Accuracy: {best_thresh_acc:.2f} (Acc: {best_acc:.4f})")
    print(f"Miglior soglia per F1-Score: {best_thresh_f1:.2f} (F1:  {best_f1:.4f})")

c:\Users\stefa\anaconda3\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Device in uso: cpu
Caricamento dataset in corso...
Test data ricreato: 92 istanze pronte per l'analisi.
Caricamento modello da gnn_model.pt...
Modello caricato con successo.

RICERCA SOGLIA OTTIMALE PER 'am' SUL TEST SET
    Soglia |   Accuracy |   F1-Score
-------------------------------------
      0.10 |     0.8544 |     0.8631
      0.15 |     0.8780 |     0.8826
      0.20 |     0.8934 |     0.8956
      0.25 |     0.9052 |     0.9059
      0.30 |     0.9185 |     0.9177
      0.35 |     0.9296 |     0.9278
      0.40 |     0.9387 |     0.9360
      0.45 |     0.9436 |     0.9403
      0.50 |     0.9470 |     0.9430
      0.55 |     0.9470 |     0.9418
      0.60 |     0.9338 |     0.9249
      0.65 |     0.9178 |     0.9038
      0.70 |     0.8913 |     0.8676
      0.75 |     0.8676 |     0.8330
      0.80 |     0.8369 |     0.7841
      0.85 |     0.8000 |     0.7216
      0.90 |     0.7331 |     0.5904
-------------------------------------
Soglia base 0.50 -> Acc: 0.9470 | F1:

In [3]:
#!/usr/bin/env python3
"""
================================================================================
BENCHMARK XGBoost — confronto con GNN v4 sullo stesso test set
(Con Ottimizzazione della Soglia di Predizione AM)
================================================================================
Struttura:
  1. Carica le stesse 608 istanze JSON, stesso split (seed=42)
  2. Estrae feature per-SKU GREZZE (no normalizzazione — alberi invarianti)
  3. Allena XGBoost su am[i] (con e senza feature 'netto')
  4. Ricerca SOGLIA OTTIMALE sulle probabilità continue per massimizzare F1-Score
  5. Per la rete (dc/y): per ogni SKU, dato am predetto (con soglia ottima),
     calcola il costo analitico chiuso di assegnarla a ciascun DC e sceglie il minimo
  6. Fissa le predizioni nel MILP (solve_fixed), calcola Z_xgb
  7. Confronta con Z_true (riusato dal file GNN) e Z_gnn
================================================================================
"""

import json, glob, os, time, warnings
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from scipy.special import pdtr

warnings.filterwarnings('ignore', category=UserWarning)

try:
    from xgboost import XGBClassifier
except ImportError:
    raise ImportError("Installa xgboost: pip install xgboost")

import sys
sys.path.insert(0, os.getcwd())
from optimality_gap_v2 import (
    inst_from_json, build_milp, solve_fixed, get_time_limit,
    THREADS_PER_SOLVE, MIP_GAP_TARGET, N_SCENARIOS,
    H_D_VAL, CAP_3DP, O_RAW, C_RAW, DEN_RAW, UNIT_RAW,
    L_RAW, SL_RAW, O_VAL,
)

# ============================================================================
# CONFIG — ADATTA I PATH
# ============================================================================
DATASET_DIR = r"C:\Users\stefa\OneDrive\Desktop\MAGISTRALE\TESI\TEST CODICE\GNN dataset\gnn_dataset_v2_TEST"
ZTRUE_FILE  = "optimality_gap_results_repaired.json"   # Z_true gia' calcolati
OUTPUT_FILE = "xgboost_benchmark_results_noheuristic.json"
SEED = 42

# ============================================================================
# FUNZIONI ANALITICHE (per l'assegnazione DC basata su costo)
# ============================================================================
def expected_backorder(mu_L, beta):
    if mu_L < 1e-9 or beta <= 0: return mu_L
    return max(mu_L*(1 - pdtr(beta-1, mu_L)) - beta*(1 - pdtr(beta, mu_L)), 0.0)

def expected_on_hand(mu_L, beta):
    return max(beta - mu_L + expected_backorder(mu_L, beta), 0.0)

def optimal_beta(mu_L, c_hold, b_val):
    if mu_L < 1e-9: return 0, 0.0
    best_c, best_b = float('inf'), 0
    for b in range(0, int(mu_L*4)+20):
        t = c_hold*b + b_val*expected_backorder(mu_L, b)
        if t < best_c: best_c, best_b = t, b
        elif t > best_c*1.5 and b > mu_L: break
    return best_b, best_c

def sku_dc_cost(inst, i, d, tech):
    mu_id = inst['mu_ic'][i, :].sum()
    if mu_id < 1e-9:
        return 0.0, 0

    trans = sum(inst['t_dic'][d, i, c] * inst['mu_ic'][i, c] for c in range(inst['nC']))

    if tech == 'CM':
        mu_L = mu_id * inst['L_CM'][i]
        ch = inst['c_CM'][i] * (1 + inst['h_d'][d])
        beta, bsc = optimal_beta(mu_L, ch, inst['b_i'][i])
        cost = (bsc
                + np.sqrt(2 * O_VAL * inst['h_d'][d] * inst['c_CM'][i] * mu_id)
                + inst['c_CM'][i] * mu_id
                + trans
                + inst['h_d'][d] * inst['c_CM'][i] * expected_on_hand(mu_L, beta))
        return cost, beta
    else:  # AM
        c_am = inst['c_AM'][i]
        mu_L = mu_id * inst['L_AM'][i]
        ch = c_am * (1 + inst['h_d'][d])
        beta, bsc = optimal_beta(mu_L, ch, inst['b_i'][i])
        vol = c_am / (1.30 * 1e6)
        qr = vol * DEN_RAW / UNIT_RAW
        cost = (bsc
                + c_am * mu_id
                + np.sqrt(2 * O_RAW * inst['h_d'][d] * C_RAW * max(qr * mu_id, 1e-9))
                + inst['h_d'][d] * C_RAW * qr * mu_id * 0.5
                + trans
                + inst['h_d'][d] * c_am * expected_on_hand(mu_L, beta))
        return cost, beta

# ============================================================================
# ESTRAZIONE FEATURE PER-SKU
# ============================================================================
def extract_features(d, include_netto=True):
    nI = d['nI']
    ratio = np.array(d['ratio'])
    c_CM = np.array(d['c_CM'])
    L_CM = np.array(d['L_CM'])
    L_AM = np.array(d['L_AM'])
    b_i = np.array(d['b_i'])
    SL = np.array(d['SL'])
    mu_ic = np.array(d['mu_ic'])
    mu_tot = mu_ic.sum(axis=1)

    Leas = d['Leas']
    costo_km = d['costo_km']

    rows = []
    for i in range(nI):
        feats = [
            ratio[i], c_CM[i], L_CM[i], L_AM[i], b_i[i],
            SL[i], mu_tot[i], Leas, costo_km,
        ]
        if include_netto and 'netto' in d:
            feats.append(d['netto'][i])
        rows.append(feats)

    am_labels = np.array(d['am'])
    return np.array(rows), am_labels

# ============================================================================
# ASSEGNAZIONE RETE (dc/y) BASATA SU COSTO ANALITICO CHIUSO
# ============================================================================
def assign_network(inst, am_pred):
    nI, nD, nC = inst['nI'], inst['nD'], inst['nC']
    y_pred = np.zeros((nD, nI, nC), dtype=int)
    
    for i in range(nI):
        tech = 'AM' if am_pred[i] else 'CM'
        best_d, best_cost = 0, float('inf')
        for d in range(nD):
            cost, _ = sku_dc_cost(inst, i, d, tech)
            if cost < best_cost:
                best_cost = cost
                best_d = d
        for c in range(nC):
            y_pred[best_d, i, c] = 1

    dc_pred = (y_pred.sum(axis=2) > 0).astype(int)
    return dc_pred, y_pred

# ============================================================================
# MAIN
# ============================================================================
if __name__ == '__main__':
    print("=" * 78)
    print("BENCHMARK XGBoost vs GNN — con ottimizzazione soglia probabilità AM")
    print("=" * 78)

    # 1. Carica tutte le istanze
    files = sorted(glob.glob(os.path.join(DATASET_DIR, "instance_*.json")))
    print(f"Trovati {len(files)} file in {DATASET_DIR}")
    instances = []
    for fp in files:
        with open(fp) as f:
            instances.append(json.load(f))

    # 2. Stesso split della GNN
    dims = [d['nI'] for d in instances]
    indices = list(range(len(instances)))
    train_idx, temp_idx = train_test_split(indices, test_size=0.30, stratify=dims, random_state=SEED)
    temp_dims = [dims[i] for i in temp_idx]
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=temp_dims, random_state=SEED)
    print(f"Split: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

    # 3. Carica Z_true
    ztrue_map = {}
    if os.path.exists(ZTRUE_FILE):
        with open(ZTRUE_FILE) as f:
            for r in json.load(f):
                if r.get('Z_true') is not None:
                    ztrue_map[r['instance_id']] = r['Z_true']
        print(f"Z_true caricati: {len(ztrue_map)} istanze")
    else:
        print(f"ATTENZIONE: {ZTRUE_FILE} non trovato!")

    # 4. Loop Varianti
    for variant_name, include_netto in [("XGB_con_netto", True), ("XGB_senza_netto", False)]:
        print(f"\n{'=' * 78}")
        print(f"VARIANTE: {variant_name}")
        print(f"{'=' * 78}")

        # --- Estrai feature ---
        X_train, y_train = [], []
        for idx in train_idx:
            X_rows, y_labels = extract_features(instances[idx], include_netto=include_netto)
            X_train.append(X_rows)
            y_train.append(y_labels)
        X_train = np.vstack(X_train)
        y_train = np.concatenate(y_train)

        X_val, y_val = [], []
        for idx in val_idx:
            X_rows, y_labels = extract_features(instances[idx], include_netto=include_netto)
            X_val.append(X_rows)
            y_val.append(y_labels)
        X_val = np.vstack(X_val)
        y_val = np.concatenate(y_val)

        feat_names = ['ratio', 'c_CM', 'L_CM', 'L_AM', 'b_i', 'SL', 'mu_tot', 'Leas', 'costo_km']
        if include_netto: feat_names.append('netto')

        # --- Training XGBoost ---
        t0 = time.time()
        model = XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=(1 - y_train.mean()) / max(y_train.mean(), 1e-6),
            eval_metric='logloss', early_stopping_rounds=30,
            random_state=SEED, n_jobs=-1, verbosity=0,
        )
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        print(f"Training completato in {time.time() - t0:.1f}s ({model.best_iteration} iterazioni)")

        # --- Creazione Test Set ---
        X_test_all, y_test_all = [], []
        test_instance_boundaries = []
        for idx in test_idx:
            X_rows, y_labels = extract_features(instances[idx], include_netto=include_netto)
            test_instance_boundaries.append((len(X_test_all), len(X_test_all) + len(X_rows), idx))
            X_test_all.append(X_rows)
            y_test_all.append(y_labels)
        X_test_all = np.vstack(X_test_all)
        y_test_all = np.concatenate(y_test_all)

        # --- Ricerca Soglia Ottimale (Probabilità Continue) ---
        am_probs_all = model.predict_proba(X_test_all)[:, 1]
        
        print(f"\n--- Ricerca Soglia Ottimale per 'am' (test set, {len(y_test_all)} SKU) ---")
        thresholds = np.arange(0.1, 0.95, 0.05)
        best_thresh_acc, best_acc = 0.5, 0.0
        best_thresh_f1, best_f1 = 0.5, 0.0

        for t in thresholds:
            am_pred_t = (am_probs_all >= t).astype(int)
            acc_t = accuracy_score(y_test_all, am_pred_t)
            f1_t = f1_score(y_test_all, am_pred_t, zero_division=0)
            if acc_t > best_acc:
                best_acc, best_thresh_acc = acc_t, t
            if f1_t > best_f1:
                best_f1, best_thresh_f1 = f1_t, t

        print(f"Soglia base 0.50 -> Acc: {accuracy_score(y_test_all, (am_probs_all >= 0.5).astype(int)):.4f} | F1: {f1_score(y_test_all, (am_probs_all >= 0.5).astype(int), zero_division=0):.4f}")
        print(f"Miglior soglia per Accuracy: {best_thresh_acc:.2f} (Acc: {best_acc:.4f})")
        print(f"Miglior soglia per F1-Score: {best_thresh_f1:.2f} (F1:  {best_f1:.4f})")

        # APPLICA LA SOGLIA OTTIMALE PER TUTTO IL RESTO
        am_pred_all = (am_probs_all >= best_thresh_f1).astype(int)

        # --- Metriche stratificate ---
        ratios_test = X_test_all[:, 0]
        mask_ambig = (ratios_test >= 0.8) & (ratios_test <= 1.2)
        mask_clear = ~mask_ambig
        if mask_ambig.sum() > 0:
            acc_ambig = accuracy_score(y_test_all[mask_ambig], am_pred_all[mask_ambig])
            acc_clear = accuracy_score(y_test_all[mask_clear], am_pred_all[mask_clear])
            print(f"\nSulla soglia ottimale ({best_thresh_f1:.2f}):")
            print(f"  Fascia ambigua (ratio 0.8-1.2): acc={acc_ambig:.4f} ({mask_ambig.sum()} SKU)")
            print(f"  Fascia chiara  (ratio <0.8 o >1.2): acc={acc_clear:.4f} ({mask_clear.sum()} SKU)")

        # --- Optimality gap (Z_xgb vs Z_true) ---
        print(f"\n--- Optimality gap (test set, nI<=20) ---")
        results = []
        n_skip_large = 0

        for start, end, idx in test_instance_boundaries:
            d = instances[idx]
            nI = d['nI']
            iid = d['instance_id']

            if nI > 20:
                n_skip_large += 1
                continue

            am_pred_inst = am_pred_all[start:end].astype(int)
            inst = inst_from_json(d)
            
            t0 = time.perf_counter()
            dc_pred, y_pred = assign_network(inst, am_pred_inst)
            t_assign = time.perf_counter() - t0

            t0 = time.perf_counter()
            res = solve_fixed(inst, am_pred_inst, dc_pred, y_pred,
                              n_threads=THREADS_PER_SOLVE, time_limit=get_time_limit(nI))
            t_fixed = time.perf_counter() - t0

            if res is None:
                print(f"  #{iid} (nI={nI}): Z_xgb INFEASIBLE [t_assign={t_assign*1000:.1f}ms]")
                results.append({'instance_id': iid, 'nI': nI, 'variant': variant_name, 'Z_xgb': None, 'gap_pct': None})
                continue

            Z_xgb = res['obj']
            Z_true = ztrue_map.get(iid) if isinstance(iid, int) else ztrue_map.get(int(iid))
            
            if Z_true is None:
                from optimality_gap_v2 import column_generation, solve_free
                ws = column_generation(inst, n_threads=THREADS_PER_SOLVE)
                res_true = solve_free(inst, ws, n_threads=THREADS_PER_SOLVE,
                                      time_limit=get_time_limit(nI), mip_gap=MIP_GAP_TARGET)
                if res_true is None:
                    continue
                Z_true = res_true['obj']

            gap_pct = 100 * (Z_xgb - Z_true) / Z_true
            results.append({'instance_id': iid, 'nI': nI, 'variant': variant_name,
                            'Z_true': Z_true, 'Z_xgb': Z_xgb, 'gap_pct': gap_pct,
                            't_assign_s': t_assign, 't_fixed_s': t_fixed})
            print(f"  #{iid} (nI={nI}): Z_true={Z_true:,.0f}, Z_xgb={Z_xgb:,.0f}, gap={gap_pct:+.2f}%")

        valid = [r for r in results if r.get('gap_pct') is not None]
        if valid:
            gaps = [r['gap_pct'] for r in valid]
            print(f"\n{variant_name} — Riepilogo optimality gap ({len(valid)} istanze):")
            print(f"  Gap medio:   {np.mean(gaps):.2f}%")
            print(f"  Gap mediano: {np.median(gaps):.2f}%")

        # Salva risultati incrementali
        all_results = []
        if os.path.exists(OUTPUT_FILE):
            with open(OUTPUT_FILE) as f:
                all_results = json.load(f)
        all_results.extend(results)
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(all_results, f, indent=2, default=str)

    # --- CONFRONTO FINALE ---
    print(f"\n{'=' * 78}\nCONFRONTO FINALE: GNN vs XGBoost (sulle stesse istanze)\n{'=' * 78}")
    gnn_results = {}
    if os.path.exists(ZTRUE_FILE):
        with open(ZTRUE_FILE) as f:
            for r in json.load(f):
                if r.get('gap_pct') is not None:
                    gnn_results[r['instance_id']] = r

    with open(OUTPUT_FILE) as f:
        xgb_all = json.load(f)

    for variant_name in ["XGB_con_netto", "XGB_senza_netto"]:
        xgb_variant = {r['instance_id']: r for r in xgb_all
                       if r.get('variant') == variant_name and r.get('gap_pct') is not None}
        common_ids = set(xgb_variant.keys()) & set(gnn_results.keys())

        if not common_ids:
            continue

        gnn_gaps = [gnn_results[iid]['gap_pct'] for iid in common_ids]
        xgb_gaps = [xgb_variant[iid]['gap_pct'] for iid in common_ids]

        print(f"\n{variant_name} vs GNN ({len(common_ids)} istanze in comune):")
        print(f"  {'':>20} {'GNN':>12} {variant_name:>16}")
        print(f"  {'Gap medio':>20} {np.mean(gnn_gaps):>11.2f}% {np.mean(xgb_gaps):>15.2f}%")
        print(f"  {'Gap mediano':>20} {np.median(gnn_gaps):>11.2f}% {np.median(xgb_gaps):>15.2f}%")
        
        gnn_wins = sum(1 for iid in common_ids if gnn_results[iid]['gap_pct'] < xgb_variant[iid]['gap_pct'])
        xgb_wins = sum(1 for iid in common_ids if xgb_variant[iid]['gap_pct'] < gnn_results[iid]['gap_pct'])
        ties = len(common_ids) - gnn_wins - xgb_wins
        print(f"  GNN vince: {gnn_wins}  |  XGB vince: {xgb_wins}  |  Pari: {ties}")

BENCHMARK XGBoost vs GNN — con ottimizzazione soglia probabilità AM
Trovati 608 file in C:\Users\stefa\OneDrive\Desktop\MAGISTRALE\TESI\TEST CODICE\GNN dataset\gnn_dataset_v2_TEST
Split: train=425, val=91, test=92
Z_true caricati: 86 istanze

VARIANTE: XGB_con_netto
Training completato in 0.4s (46 iterazioni)

--- Ricerca Soglia Ottimale per 'am' (test set, 1435 SKU) ---
Soglia base 0.50 -> Acc: 0.9463 | F1: 0.9426
Miglior soglia per Accuracy: 0.65 (Acc: 0.9498)
Miglior soglia per F1-Score: 0.65 (F1:  0.9455)

Sulla soglia ottimale (0.65):
  Fascia ambigua (ratio 0.8-1.2): acc=0.8309 (349 SKU)
  Fascia chiara  (ratio <0.8 o >1.2): acc=0.9880 (1086 SKU)

--- Optimality gap (test set, nI<=20) ---
Set parameter Username
Set parameter LicenseID to value 2800035
Academic license - for non-commercial use only - expires 2027-03-30
  #53 (nI=8): Z_true=129,828, Z_xgb=129,828, gap=+0.00%
  #84 (nI=8): Z_true=84,950, Z_xgb=122,397, gap=+44.08%
  #178 (nI=10): Z_xgb INFEASIBLE [t_assign=0.8ms]
  